In [ ]:
#Leo Murch
#Practicum 5
#ast4762
#10/19/26

In [21]:
#import everything from hw7, hw6:
print("Problem 1")
from hw6_aleonav import *

Problem 1


In [22]:
#normmedcomb:


print("Problem 2")
def normmedcomb(data, corner_coords = None):
    """
    Median-combine a stack of dark-subtracted frames after normalizing each one.

    Each frame is divided by the median of its normalization region so frames
    with different sky levels can be combined. The input array is not modified.

    Parameters
    ----------
    3darr : array
        3D array of dark-subtracted sky flux, shaped (nframes, ny, nx).
    corner_coords : tuple of tuple, optional
        Corners of the region used to compute each frame's normalization
        factor, given as ((y1, x1), (y2, x2)). Defaults to the whole frame.

    Returns
    -------
    combined : array
        Normalized, median-combined 2D image.
    factors : array
        1D array of the normalization factor for each frame.

    Examples
    --------
    >>> combined, factors = normmedcomb(obj_darksub,
    ...                                 ((225, 225), (-225, -225)))
    >>> print(factors)
    [1.02 0.98 1.01 ...]

    >>> combined, factors = normmedcomb(obj_darksub)
    >>> combined.shape
    (1024, 1024)
    """
    
    #check to see if the corner_coords is given or not:
    if corner_coords is None:
        region = data #region is the whole array, not to modify the data
    else:
        (y1, x1), (y2, x2) = corner_coords #unpack the given tuple
        region = data[:, y1:y2, x1:x2] #all images, from y1-y2, and x1-x2 only is the region.

    norm_factor = np.median(region, axis=(1,2)) #to take the avg of the image, not along the actual # of images
    scaled_data = data / norm_factor[:, None, None] #axis = 0
    med_combined = np.median(scaled_data, axis = 0) #same as above along each image

    return med_combined, norm_factor

Problem 2


In [ ]:
print('Problem 3')
#do this here since we didn't actually do it in hw6
objsubdark = objdata - dark_median
region = ((225, 225), (-225, -225)) #FIXED! wrong in P5 and good now, i'll resubmit

#call function:
sky_mednorm, norm_factors = normmedcomb(objsubdark, region)

#write to a fits file:
fits.writeto('sky_13s_mednorm.fits', sky_mednorm, overwrite = True)

#NOTE
#i didnt put an overwrite in this one, so just add:
#  overwrite = True
#if you wanna try to reproduce the data again yourself.


Problem 3


In [ ]:
print(norm_factors)
#[10864. 11057. 10536. 10559. 11046. 10623.  9916.  9916.  9859.]

[10864. 11057. 10536. 10559. 11046. 10623.  9916.  9916.  9859.]


In [25]:
#Leo Murch
#HW8
#ast4762
#10/19/26

In [26]:
#homework 8 portion of the hw8 py file.

In [29]:
print('Problem 2')
#this one now takes two 2D arrays
#the scaling back up to get our stars back that we talked abt on friday (kinda glad i waited!)
def skycormednorm (objarr, normarr, corner_coords = None):
    """
    Subtract a normalized sky frame from one dark-subtracted object frame.

    The median of the object's normalization region is used to scale the sky
    back up, then that sky is subtracted. The input arrays are not modified.

    Parameters
    ----------
    objarr : array
        Dark-subtracted 2D object frame.
    normarr : array
        Normalized 2D sky frame (e.g. from normmedcomb).
    corner_coords : tuple of tuple, optional
        Corners of the region used to compute the object's normalization
        factor, given as ((y1, x1), (y2, x2)). Defaults to the whole frame.

    Returns
    -------
    nosky : array
        Sky-subtracted 2D object frame.

    Examples
    --------
    >>> nosky = skycormednorm(obj_darksub[0], sky_mednorm,
    ...                       ((225, 225), (-225, -225)))
    >>> nosky.shape
    (1024, 1024)

    >>> nosky = skycormednorm(obj_darksub[0], sky_mednorm)
    >>> nosky[217, 184]
    12.3
    """
    #check like last time to see if the region is specified:
    if corner_coords is None:
        region = objarr
    else:
        #unpack tuple like last time:
        (y1, x1),(y2, x2) = corner_coords
        region = objarr[y1:y2, x1:x2] #within the 2d arr, at the wanted coords
    
    factor = np.median(region) #to multiply to get de-normalized
    return objarr - (factor * normarr) #sub from object frame, return result

Problem 2


In [30]:
#now apply it w/ a loop to all of the object data:
noskyframe = [] #to populate
for i in range(objsubdark.shape[0]): #0 is for each image #
    noskyframe.append(skycormednorm(objsubdark[i], sky_mednorm, region))

#and write the last one to a fits file:
fits.writeto('stars_13s_9_nosky.fits', noskyframe[-1], overwrite = True)
print(f"\nThe value of pixel [217, 184] of stars_13s_9_nosky.fits is:\n{noskyframe[-1][217, 184]}")


The value of pixel [217, 184] of stars_13s_9_nosky.fits is:
-97.39643001210243
